In [1]:
import qrcode
from qrcode.constants import ERROR_CORRECT_H
from PIL import Image

In [50]:
def build_mecard(
    first_name: str,
    last_name: str,
    phone: str | None = None,
    email: str | None = None,
    company: str | None = None,
    title: str | None = None,
    website: str | None = None,
    address: str | None = None
) -> str:
    """
    Génère un payload MECARD compact pour QR code contact.
    """

    fields = []

    # format MECARD: NOM,PRENOM
    fields.append(f"N:{last_name},{first_name}")

    if phone:
        fields.append(f"TEL:{phone}")

    if email:
        fields.append(f"EMAIL:{email}")

    if company:
        fields.append(f"ORG:{company}")

    if title:
        fields.append(f"NOTE:{title}")

    if website:
        fields.append(f"URL:{website}")

    if address:
        fields.append(f"ADR:{address}")

    payload = "MECARD:" + ";".join(fields) + ";;"

    return payload

In [ ]:
def build_vcard(
    first_name: str,
    last_name: str,
    phone: str | None = None,
    email: str | None = None,
    company: str | None = None,
    title: str | None = None,
    website: str | None = None,
    address: str | None = None
) -> str:
    """
    Génère une vCard 3.0 compatible iOS / Android.
    """

    vcard_lines = [
        "BEGIN:VCARD",
        "VERSION:3.0",
        f"N:{last_name};{first_name};;;",
        f"FN:{first_name} {last_name}",
    ]

    if company:
        vcard_lines.append(f"ORG:{company}")

    if title:
        vcard_lines.append(f"TITLE:{title}")

    if phone:
        vcard_lines.append(f"TEL;TYPE=CELL:{phone}")

    if email:
        vcard_lines.append(f"EMAIL;TYPE=INTERNET:{email}")

    if website:
        vcard_lines.append(f"URL:{website}")

    if address:
        # Format ADR: PO Box;Extended;Street;City;Region;PostalCode;Country
        vcard_lines.append(f"ADR;TYPE=WORK:;;{address};;;;")

    vcard_lines.append("END:VCARD")

    return "\n".join(vcard_lines)


def generate_qr(
    payload: str,
    output_path: str = "contact_qr.png",
    logo_path: str | None = None,
    box_size: int = 12,
    border: int = 4
):

    qr = qrcode.QRCode(
        version=None, # fixe la densité
        error_correction=ERROR_CORRECT_H,
        box_size=box_size,
        border=border,
    )

    qr.add_data(payload)
    qr.make(fit=True)

    qr_img = qr.make_image(fill_color="black", back_color="white").convert("RGB")

    if logo_path:
        logo = Image.open(logo_path).convert("RGBA")

        qr_w, qr_h = qr_img.size
        logo_size = qr_w // 5
        logo = logo.resize((logo_size, logo_size), Image.LANCZOS)

        pos = ((qr_w - logo_size) // 2, (qr_h - logo_size) // 2)
        qr_img.paste(logo, pos, mask=logo)

    qr_img.save(output_path)
    print(f"QR contact généré : {output_path}")

In [48]:
import qrcode
from qrcode.constants import ERROR_CORRECT_H
from qrcode.image.styledpil import StyledPilImage
from qrcode.image.styles.colormasks import SolidFillColorMask
from PIL import Image

from qrcode.image.styles.moduledrawers import (
    RoundedModuleDrawer,
    CircleModuleDrawer,
    VerticalBarsDrawer,
    HorizontalBarsDrawer
)


def generate_qr(payload, output_path="qr.png", logo_path=None):

    qr = qrcode.QRCode(
        version=12, # fixe la densité
        error_correction=ERROR_CORRECT_H,
        box_size=12,
        border=4,
    )

    qr.add_data(payload)
    qr.make(fit=False) #True

    img = qr.make_image(
        image_factory=StyledPilImage,
        module_drawer=RoundedModuleDrawer(), #RoundedModuleDrawer,CircleModuleDrawer,VerticalBarsDrawer,HorizontalBarsDrawer
        color_mask=SolidFillColorMask(
            front_color=(0,0,0),
            back_color=(255,255,255)
        )
    ).convert("RGB")

    if logo_path:
        logo = Image.open(logo_path).convert("RGBA")

        qr_w, qr_h = img.size
        logo_size = qr_w // 5
        logo = logo.resize((logo_size, logo_size), Image.LANCZOS)

        pos = ((qr_w - logo_size) // 2, (qr_h - logo_size) // 2)
        img.paste(logo, pos, mask=logo)

    img.save(output_path)

In [ ]:
vcard = build_vcard(
first_name="Stéphane",
last_name="MESLÉ",
phone="+33644723935",
# email="stephane@mesle-entreprises.fr",
# company="Entreprise Stéphane MESLÉ VTC",
# title="Location de voiture avec chauffeur privé",
website="https://www.mesle-entreprises.fr/",
)

generate_qr(
payload=vcard,
output_path="contact_qr_vcard.png",
logo_path=None,  # mettre None si pas de logo
)

In [ ]:
payload = build_mecard(
    first_name="John",
    last_name="Doe",
    phone="+33123456789",
    email="john@company.com",
    company="SpaceCorp",
    website="https://spacecorp.com"
)

generate_qr(payload, "contact_qr_mecard.png")